In [1]:
!pip install transformers datasets torch accelerate

In [2]:
import torch
from datasets import Dataset
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

In [3]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT2 does not have padding token by default
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [4]:
stories = [
    "Once upon a time in a quiet village, a young girl discovered a hidden door in the forest.",
    "The robot looked at the stars and wondered why humans loved stories so much.",
    "In a distant galaxy, a lonely astronaut received a mysterious signal from an unknown planet.",
    "A curious cat followed a glowing butterfly into a magical garden.",
    "The old library contained a book that could change the future."
]

dataset = Dataset.from_dict({"text": stories})

In [5]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [6]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [7]:
!pip install --upgrade transformers

In [10]:
training_args = TrainingArguments(
    output_dir="./gpt2_story_model",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    save_steps=500,
    logging_steps=50,
    report_to="none"
)

In [11]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [12]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15, training_loss=2.6231396993001304, metrics={'train_runtime': 48.6443, 'train_samples_per_second': 0.514, 'train_steps_per_second': 0.308, 'total_flos': 816537600000.0, 'train_loss': 2.6231396993001304, 'epoch': 5.0})

In [13]:
trainer.save_model("./gpt2_story_model")
tokenizer.save_pretrained("./gpt2_story_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2_story_model/tokenizer_config.json',
 './gpt2_story_model/tokenizer.json')

In [14]:
prompt = "Once upon a time"

inputs = tokenizer(prompt, return_tensors="pt")

output = model.generate(
    inputs["input_ids"],
    max_length=80,
    num_return_sequences=1,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    do_sample=True
)

generated_story = tokenizer.decode(output[0], skip_special_tokens=True)

print(generated_story)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Once upon a time he dreamed of traveling through time, and space. He imagined that every thing that he saw would ever be a place for him. It was in the deep, quiet of the mountains, and he imagined that his soul was the place that could hold people together.

His soul had returned to Earth. He returned to his home planet. He had returned back to the land.
